# MLP Memory Footprint Calculator

Calculate memory footprint for inference and training (minimal SGD, no RMSProp).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_memory_footprint(layer_sizes, verbose=True):
    """
    Calculate memory footprint for a static MLP (float32).
    
    Parameters:
    -----------
    layer_sizes : list of int
        Architecture as [input_dim, hidden1_dim, hidden2_dim, ..., output_dim]
    verbose : bool
        Print detailed breakdown
    
    Returns:
    --------
    dict with keys: 'inference_elements', 'inference_bytes', 'inference_kb',
                    'training_elements', 'training_bytes', 'training_kb',
                    'W', 'B', 'I', 'O', 'M'
    """
    
    # Layer sizes
    s = np.array(layer_sizes, dtype=np.int64)
    L = len(s) - 1  # number of layers
    
    # Calculate quantities
    # W = sum of all weights (each layer i: s[i] * s[i+1])
    W = sum(s[i] * s[i+1] for i in range(L))
    
    # B = sum of all output dimensions (s[1] + s[2] + ... + s[L])
    B = sum(s[1:])
    
    # I = sum of all input dimensions (s[0] + s[1] + ... + s[L-1])
    I = sum(s[:-1])
    
    # O = network output dimension
    O = s[L]
    
    # M = maximum layer width
    M = max(s)
    
    # Formulas (in number of float32 elements)
    inf_elements = W + 2*B + 2*M
    train_elements = 2*W + 2*I + 4*B + 4*M + 2*O
    
    # Convert to bytes (4 bytes per float32)
    bytes_per_element = 4
    inf_bytes = inf_elements * bytes_per_element
    train_bytes = train_elements * bytes_per_element
    
    # Convert to KB
    inf_kb = inf_bytes / 1024
    train_kb = train_bytes / 1024
    
    if verbose:
        print("=" * 70)
        print(f"MLP Architecture: {' → '.join(map(str, layer_sizes))}")
        print(f"Number of layers: {L}")
        print("=" * 70)
        print()
        print("Quantities:")
        print(f"  W (total weight count):        {W:12,d} elements")
        print(f"  B (sum layer outputs):         {B:12,d} elements")
        print(f"  I (sum layer inputs):          {I:12,d} elements")
        print(f"  O (output dimension):          {O:12,d} elements")
        print(f"  M (maximum layer width):       {M:12,d} elements")
        print()
        print("INFERENCE (no training):")
        print(f"  Formula: M_inf = W + 2B + 2M")
        print(f"           M_inf = {W} + 2*{B} + 2*{M}")
        print(f"           M_inf = {W} + {2*B} + {2*M}")
        print(f"  Elements:  {inf_elements:12,d}")
        print(f"  Bytes:     {inf_bytes:12,d}  ({inf_kb:8.2f} KB)")
        print()
        print("TRAINING (minimal SGD, no RMSProp):")
        print(f"  Formula: M_train = 2W + 2I + 4B + 4M + 2O")
        print(f"           M_train = 2*{W} + 2*{I} + 4*{B} + 4*{M} + 2*{O}")
        print(f"           M_train = {2*W} + {2*I} + {4*B} + {4*M} + {2*O}")
        print(f"  Elements:  {train_elements:12,d}")
        print(f"  Bytes:     {train_bytes:12,d}  ({train_kb:8.2f} KB)")
        print()
        print(f"Training overhead: {train_bytes - inf_bytes:,d} bytes ({train_kb - inf_kb:.2f} KB)")
        print(f"Training/Inference ratio: {train_bytes / inf_bytes:.2f}x")
        print("=" * 70)
    
    return {
        'W': W, 'B': B, 'I': I, 'O': O, 'M': M,
        'inference_elements': inf_elements,
        'inference_bytes': inf_bytes,
        'inference_kb': inf_kb,
        'training_elements': train_elements,
        'training_bytes': train_bytes,
        'training_kb': train_kb,
    }

# Example from the user
# Input: 64; Hidden: 64, 32, 32, 10; Output: 8
architecture = [64, 64, 32, 32, 10, 8]
result = calculate_memory_footprint(architecture, verbose=True)

## Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['Elements', 'Bytes', 'KB'],
    'Inference': [
        f"{result['inference_elements']:,d}",
        f"{result['inference_bytes']:,d}",
        f"{result['inference_kb']:.2f}"
    ],
    'Training': [
        f"{result['training_elements']:,d}",
        f"{result['training_bytes']:,d}",
        f"{result['training_kb']:.2f}"
    ]
})

print(summary.to_string(index=False))

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: KB comparison
ax = axes[0]
categories = ['Inference', 'Training']
kb_values = [result['inference_kb'], result['training_kb']]
colors = ['#2E86AB', '#A23B72']

bars = ax.bar(categories, kb_values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Memory (KB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Footprint Comparison', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar, val in zip(bars, kb_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f} KB', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart: Training breakdown
ax = axes[1]
train_bytes = result['training_bytes']
components = {
    '2W (weights+gradients)': 2 * result['W'],
    '2I (inputs cached)': 2 * result['I'],
    '4B (outputs+caches)': 4 * result['B'],
    '4M (buffers)': 4 * result['M'],
    '2O (loss)': 2 * result['O'],
}

sizes = list(components.values())
labels = [f"{k}\n({v:,d})" for k, v in components.items()]
colors_pie = ['#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4749']

ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax.set_title('Training Memory Breakdown\n(in elements)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('mlp_memory_footprint.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: mlp_memory_footprint.png")

## Try Different Architectures

In [ ]:
# Compare different architectures
architectures = [
    ([64, 64, 32, 32, 10, 8], "Original (64→64→32→32→10→8)"),
    ([64, 128, 64, 32, 8], "Wider middle (64→128→64→32→8)"),
    ([64, 32, 16, 8], "Smaller (64→32→16→8)"),
]

results_table = []
for arch, name in architectures:
    r = calculate_memory_footprint(arch, verbose=False)
    results_table.append({
        'Architecture': name,
        'Inf (KB)': f"{r['inference_kb']:.2f}",
        'Train (KB)': f"{r['training_kb']:.2f}",
        'Ratio': f"{r['training_kb'] / r['inference_kb']:.2f}x"
    })

df = pd.DataFrame(results_table)
print("\nComparison across architectures:")
print(df.to_string(index=False))

## Formula Reference

**Quantities:**
- `W` = total weight count = ∑ᵢ sᵢ · sᵢ₊₁
- `B` = sum of layer outputs = ∑ⱼ sⱼ (for j=1 to L)
- `I` = sum of layer inputs = ∑ᵢ sᵢ (for i=0 to L-1)
- `O` = output dimension = s_L
- `M` = maximum layer width = max(s₀, s₁, ..., s_L)

**Memory Formulas (in float32 elements):**
- **Inference:** `M_inf = W + 2B + 2M`
- **Training (minimal SGD, no RMSProp):** `M_train = 2W + 2I + 4B + 4M + 2O`

**To convert to bytes:** multiply by 4 (size of float32)

**Training includes:**
- Weight gradients, bias gradients, cached inputs, input gradients
- Forward & backward ping-pong buffers
- Output cache and loss gradient cache
- **Excludes:** RMSProp second-moment buffers (m²_accum)
